In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

spark.version  # confirm Spark is running

'3.5.0'

In [7]:
import os

API_BASE_URL = os.environ.get("API_BASE_URL", "http://mock-api:8000")
API_USERNAME = os.environ.get("API_USERNAME", "candidate")
API_PASSWORD = os.environ.get("API_PASSWORD", "blue-owls-2026")

In [8]:
import requests

# Authenticate
resp = requests.post(f"{API_BASE_URL}/api/v1/auth/token", json={
    "username": API_USERNAME,
    "password": API_PASSWORD
})
token = resp.json()["access_token"]

# Fetch first page of orders
headers = {"Authorization": f"Bearer {token}"}
data = requests.get(f"{API_BASE_URL}/api/v1/data/orders", headers=headers, params={"page": 1, "page_size": 1000})
print(data.json()["pagination"])

{'page': 1, 'page_size': 1000, 'total_records': 99441, 'total_pages': 100, 'has_next': True}


In [9]:
import requests
import pandas as pd
import time
from datetime import datetime

In [10]:
BASE_URL = "http://mock-api:8000"
USERNAME = "candidate"
PASSWORD = "blue-owls-2026"   # try 2026 first, if fails use 2025

In [11]:
def get_token():
    res = requests.post(f"{BASE_URL}/api/v1/auth/token", json={
        "username": USERNAME,
        "password": PASSWORD
    })

    if res.status_code != 200:
        raise Exception(f"Auth failed: {res.text}")

    return res.json()["access_token"]

In [12]:
def fetch_data(endpoint):
    token = get_token()
    page = 1
    all_data = []

    while True:
        headers = {"Authorization": f"Bearer {token}"}
        params = {"page": page, "page_size": 1000}

        res = requests.get(f"{BASE_URL}/api/v1/data/{endpoint}", headers=headers, params=params)

        if res.status_code == 401:
            token = get_token()
            continue

        if res.status_code in [429, 500]:
            time.sleep(2)
            continue

        data = res.json()
        records = data.get("data", [])

        if not records:
            break

        all_data.extend(records)

        if not data["pagination"]["has_next"]:
            break

        page += 1

    return all_data

In [15]:
import os  # make sure this is imported

def save_bronze(data, endpoint):
    df = pd.DataFrame(data)
    df["_ingested_at"] = datetime.now()
    df["_source"] = endpoint

    path = "/home/jovyan/work/output/bronze"

    os.makedirs(path, exist_ok=True)   # 🔥 THIS LINE FIXES EVERYTHING

    df.to_csv(f"{path}/{endpoint}.csv", index=False)

    print(f"Saved {endpoint}")

In [16]:
endpoints = ["orders", "order_items", "customers", "products", "sellers", "payments"]

for ep in endpoints:
    print(f"Fetching {ep}...")
    data = fetch_data(ep)
    save_bronze(data, ep)

Fetching orders...
Saved orders
Fetching order_items...
Saved order_items
Fetching customers...
Saved customers
Fetching products...
Saved products
Fetching sellers...
Saved sellers
Fetching payments...
Saved payments
